# Ontology mapping

This notebook is the high-level user workspace for Step 3.

Goal:
- review the main input and config paths
- run the canonical exporter
- inspect the summary counts and warnings
- keep implementation logic in `3_ontology_mapping/scripts/generator_core.py`

Mapping data lives in `3_ontology_mapping/config/*.yaml`, while the Markdown note in that folder is support documentation only.


## Workflow

1. Set the motel-db input path and TTL output path.
2. Import `write_ttl_output(...)` and config-path constants from `generator_core.py`.
3. Confirm which YAML files are driving the runtime mapping.
4. Run the export.
5. Review the summary counts and warnings.

The notebook intentionally does **not** duplicate generator logic or mapping tables.


In [1]:
from pathlib import Path

PATH_MOTEL_DB = Path(r"..\motel-db")
OUTPUT_TTL = Path(r"..\3_ontology_mapping\output_ttl\cls_atr_motel.ttl")

PATH_MOTEL_DB, OUTPUT_TTL

(WindowsPath('../motel-db'),
 WindowsPath('../3_ontology_mapping/output_ttl/cls_atr_motel.ttl'))

## Import Shared Generator

This cell makes sure the repository root is importable, then loads the canonical export function and config paths.


In [2]:
import sys

repo_root = Path.cwd()
if not (repo_root / "3_ontology_mapping").exists():
    for candidate in [repo_root, *repo_root.parents]:
        if (candidate / "3_ontology_mapping").exists():
            repo_root = candidate
            break

repo_root_str = str(repo_root)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

scripts_dir = repo_root / "3_ontology_mapping" / "scripts"
scripts_dir_str = str(scripts_dir)
if scripts_dir_str not in sys.path:
    sys.path.insert(0, scripts_dir_str)

from generator_core import DEFAULT_ATTRIBUTE_MAPPING_CONFIG
from generator_core import DEFAULT_UNIT_MAPPING_CONFIG
from generator_core import write_ttl_output


## Runtime Config

These YAML files are the machine-readable mapping inputs used by the shared generator.


In [ ]:
{
    "attribute_mapping": str(DEFAULT_ATTRIBUTE_MAPPING_CONFIG),
    "unit_mappings": str(DEFAULT_UNIT_MAPPING_CONFIG),
    "support_notes": str(repo_root / "3_ontology_mapping" / "config" / "attribute_ontology_linkage_audit.md"),
}


## Run Export

This writes the canonical TTL file and returns a small summary dictionary.


In [4]:
result = write_ttl_output(
    path_motel_db=PATH_MOTEL_DB,
    output_ttl=OUTPUT_TTL,
    generated_by="3_ontology_mapping.ipynb",
)

ttl = result["ttl"]
generated_at = result["generated_at"]
warnings = result["warnings"]
stats = result["stats"]
config_paths = result.get("config_paths", {})

summary = {
    "generated_at": generated_at,
    "output_ttl": str(OUTPUT_TTL),
    "attribute_mapping": str(config_paths.get("attribute_mapping", DEFAULT_ATTRIBUTE_MAPPING_CONFIG)),
    "unit_mappings": str(config_paths.get("unit_mapping", DEFAULT_UNIT_MAPPING_CONFIG)),
    "n_lines": ttl.count("\n"),
    "n_triple_ends_approx": ttl.count(" ."),
    "n_instances_with_attributes_approx": ttl.count("dici_onto:hasAttribute"),
    "flows": stats.get("flows", 0),
    "embedded_carbon": stats.get("embedded_carbon", 0),
    "warnings": len(warnings),
}

summary


["LE_00100: unmapped attribute 'Storage Carrier'",
 "LE_00100: unmapped attribute 'Minimum Installation'",
 "LE_00100: unmapped attribute 'ChargingCapacityFactor'",
 "LE_00100: unmapped attribute 'Discharging Capacity Factor'",
 "LE_00100: unmapped attribute 'Charging Efficiency'",
 "LE_00100: unmapped attribute 'Discharging Efficiency'",
 "LE_00100: unmapped attribute 'Minimum Soc'",
 "LE_00100: unmapped attribute 'Maximum Soc'",
 "LE_00100: unmapped attribute 'Standby Loss Per Hour'",
 "LE_00100: unmapped attribute 'Capital Expenditure per Storage Capacity'",
 "LE_00100: unmapped attribute 'Operational Expenditure Per Storage Capacity'",
 "LE_00101: unmapped attribute 'Storage Carrier'",
 "LE_00101: unmapped attribute 'Minimum Installation'",
 "LE_00101: unmapped attribute 'ChargingCapacityFactor'",
 "LE_00101: unmapped attribute 'Discharging Capacity Factor'",
 "LE_00101: unmapped attribute 'Charging Efficiency'",
 "LE_00101: unmapped attribute 'Discharging Efficiency'",
 "LE_00101:

## Inspect Warnings

Warnings usually mean one of these:
- a motel-db attribute is not yet mapped to an ontology class
- a carrier lookup is missing
- an embedded-carbon record could not be linked to a technology-year instance
- a YAML config entry is incomplete or inconsistent

Showing only the first entries keeps the notebook light.
